# Cardio, HELOC, and Market: Five-Run GLOBE-CE Audit

This notebook leaves the German Credit experiment files unchanged and applies the same GLOBE-CE procedure to the other three datasets. Each dataset uses the original experiment's data split, standardization, MLP, 100 explained samples, and seeds `[40, 41, 42, 43, 44]`.

DCE and CDCE SWD/WD values are read directly from the existing five-run Excel files in the repository, without rerunning or overwriting them. GLOBE-CE outputs are audited against the four institutional constraints and data-domain validity requirements.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "explainers").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the repository root.")

ROOT = find_repo_root()
BASELINE_DIR = ROOT / "baselines" / "globe-ce"
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(BASELINE_DIR))

from multi_dataset_globe_audit import (
    DATASET_CONFIGS,
    SEEDS,
    run_all,
)

pd.set_option("display.max_columns", None)
OUTPUT_DIR = BASELINE_DIR / "results" / "multi-dataset"
print(f"Datasets: {list(DATASET_CONFIGS)}")
print(f"Seeds: {SEEDS}")
print(f"Output directory: {OUTPUT_DIR}")

## Evaluation Protocol

GLOBE-CE resamples the global transformation direction in each run. SWD uses seed-fixed random projections for that run. WD follows the saved convention of the three original repeated-run scripts and is not square-rooted again at the outermost level. The favorable class is `Response=1` for Market and class 0 for Cardio and HELOC. Mean is computed on the original feature scale. Consistent with the original DCE/CDCE validation notebooks, Std. uses the sample standard deviation (`ddof=1`). LSC uses the dataset-specific linear relation estimated from the training data, and FSD uses the same empirical-CDF violation-count definition.

The validity metrics do not repair or clip baseline outputs. They only check whether changed samples fall outside the training-data range or introduce fractional values in originally integer-valued fields.

In [ ]:
globe_runs, globe_summary, alignment_runs, alignment_summary = run_all(
    root=ROOT,
    output_dir=OUTPUT_DIR,
)
print("All five-run experiments completed.")

## GLOBE-CE: Per-Run Results

The table below retains coverage, SWD, WD, the four constraint metrics, and data-domain validity metrics for each seed, making it possible to identify anomalous runs behind the averages.

In [ ]:
display(globe_runs.round(5))

## GLOBE-CE: Five-Run Mean ± Standard Deviation

Means are used in the main paper tables, while standard deviations are reported in parentheses or with `±`. Both the mean and standard deviation are retained for count metrics to reflect stability across global directions.

In [ ]:
display(globe_summary.round(5))

## Comparison of DCE, CDCE, and GLOBE-CE

DCE/CDCE values come from the existing five-run Excel results, while GLOBE-CE values come from this notebook. CDCE is listed separately for Mean, Std., LSC, and FSD because the four variants optimize different institutional constraints.

In [ ]:
comparison_columns = [
    "Dataset", "Method", "SWD_mean", "SWD_std", "WD_mean", "WD_std"
]
display(alignment_summary[comparison_columns].round(5))

## Output Files

The experiment generates a per-run CSV, a mean/standard-deviation CSV, and an Excel workbook containing all results and 15 counterfactual sets. The original Cardio, HELOC, Market, and German Credit files are not overwritten.

In [ ]:
for path in sorted(OUTPUT_DIR.iterdir()):
    print(path.name)